# 03 · Je eigen server inrichten

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Tot nu toe bewaar je de databank op je laptop en in Codespaces. Om straks gegevens met een andere gebruiker uit te wisselen, richt je een eigen server in die altijd via internet bereikbaar blijft. Die server huur je via jullie teamaccount bij DigitalOcean. Je installeert er een **MQTT-broker** op (die berichten doorgeeft) en een beveiligde **API-testdienst**.

Je maakt eerst de server aan, logt in met een wachtwoord en installeert de nodige software. Daarna test je vanuit Codespaces of er veilig een bericht heen en terug kan en of de beveiligde webverbinding werkt.

**Nodig:** je bestaande Codespace, het bestand `01_werkplekcontrole.json` uit notebook 01, toegang tot het DigitalOcean-team **lanupro-tech** en het project **starten-met-iot**.

Voer de Python-cellen van boven naar beneden uit en stop tussendoor voor de handmatige stappen. De opdrachten in de losse grijze `bash`-cellen typ of plak je in een terminal. Boven elk stuk staat steeds aangegeven waar je de opdracht uitvoert (op de DigitalOcean-website, op je server of in Codespaces).

</div>

<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Voor je begint</strong><br>Gebruik team <strong>lanupro-tech</strong> en project <strong>starten-met-iot</strong>. Maak één eigen Droplet met Debian 13 en het goedkoopste Basic/Regular-plan. Kies wachtwoordaanmelding en laat Startup scripts uitgeschakeld. Hieronder doorloop je alle instellingen stap voor stap.</div>

## 1. Je eigen werkplek ophalen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 03)

Open `notebooks/03_eigen_server_inrichten.ipynb` en kies rechtsboven opnieuw **Python 3.12** als kernel. De cel hieronder haalt jouw opgeslagen instellingen uit notebook 01 opnieuw op, zodat je gebruikerscode en vaste servernaam meteen kloppen.

</div>

In [2]:
from pathlib import Path
import subprocess
import json
import re
import sys
import ipaddress

resultaat = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
if resultaat.returncode:
    raise RuntimeError("Open notebook 03 vanuit de practicumrepository in Codespaces.")
hoofdmap = Path(resultaat.stdout.strip())
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=hoofdmap, text=True).strip()
if not re.fullmatch(r"student/[a-z0-9][a-z0-9_-]*", branch):
    raise RuntimeError("Selecteer je eigen student/voornaam_achternaam-branch uit notebook 01.")
instellingenpad = hoofdmap / "gegevens/inlevering/01_werkplekcontrole.json"
if not instellingenpad.is_file():
    raise FileNotFoundError("Bewaar eerst de werkplekcontrole uit notebook 01.")
instellingen = json.loads(instellingenpad.read_text(encoding="utf-8"))
deelnemer = instellingen.get("deelnemerscode", "")
if instellingen.get("branch") != branch or deelnemer != branch.removeprefix("student/"):
    raise ValueError("Je opgeslagen instellingen horen niet bij deze branch. Controleer notebook 01.")
if not re.fullmatch(r"[a-z0-9][a-z0-9_-]{0,62}", deelnemer):
    raise ValueError("Gebruik een deelnemerscode van maximaal 63 tekens, met kleine letters, cijfers, _ of -.")
werkmap = hoofdmap / "gegevens/werk"
werkmap.mkdir(parents=True, exist_ok=True)
print("Deelnemerscode:", deelnemer)
print("Servernaam:", "iot-" + deelnemer.replace("_", "-"))
print("Team: lanupro-tech | Project: starten-met-iot")

Deelnemerscode: kenzo_meulemeester
Servernaam: iot-kenzo-meulemeester
Team: lanupro-tech | Project: starten-met-iot


## 2. Wat huur je bij DigitalOcean?

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Een **online server** (bij DigitalOcean een **Droplet** genoemd) is een virtuele computer in een datacenter die dag en nacht blijft draaien en via internet bereikbaar is, zolang je hem niet uitschakelt en er geen storing optreedt. Daardoor is een server de ideale plek voor programma's en taken die continu actief moeten blijven, terwijl jij je eigen laptop gewoon kunt afsluiten. Een voorbeeld van zo'n continu programma is een **MQTT-broker** (zoals **Mosquitto**, dat we straks installeren), die op elk moment metingen moet kunnen ontvangen en doorsturen.

**Debian** is het besturingssysteem op die server (vergelijkbaar met Windows of macOS, en verwant aan andere Linux-varianten zoals Ubuntu). Door allemaal **Debian 13** te kiezen, werken dezelfde installatieopdrachten bij iedereen.

Drie begrippen die steeds terugkomen:
- **Hostname:** de herkenbare naam van jouw server (zoals `iot-jan-janssens`).
- **Publiek IPv4-adres:** het vaste internetadres waarmee Codespaces jouw server bereikt.
- **Poort:** een soort deurnummer dat bepaalt welk programma op de server een verbinding afhandelt. We gebruiken poort **22** voor beheer (SSH), **8883** voor beveiligd MQTT en **443** voor HTTPS.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 1.** Waar draait Python wanneer je een notebookcel uitvoert, en waar draaien de continue serverprogramma's die je straks installeert (zoals een MQTT-broker)? Wat gebeurt er met die serverprogramma's als je jouw eigen laptop afsluit?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 1):* Python draait in github, de serverprogrammas draaien in DigitalOcean, als je de laptop afsluit, dan is de server niet meer beschikbaar voor andere apparaten*

Voeg hier je antwoord in. Benoem de computers afzonderlijk.

</div>

## 3. Het juiste team en project kiezen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op de DigitalOcean-website

Voor dit practicum is er bij DigitalOcean een centraal team aangemaakt (**`lanupro-tech`**), waarvoor je een uitnodiging hebt ontvangen. Door die uitnodiging te aanvaarden en binnen dat team te werken, kun je jouw eigen Droplet aanmaken terwijl de kosten rechtstreeks door de universiteit worden betaald. Binnen dat team staan alle practicumservers samen in het project **`starten-met-iot`**.

1. Aanvaard eerst de uitnodiging voor team **lanupro-tech** (je kunt met GitHub aanmelden, maar de teamuitnodiging blijft een aparte stap).
2. Open [cloud.digitalocean.com](https://cloud.digitalocean.com/), klik op je profiel, kies **Switch Teams** en selecteer **lanupro-tech**.
3. Open het project **starten-met-iot** en controleer dat zowel het team als het project klopt.
4. Zie je het team of project niet staan? Laat je toegangsrechten nakijken en maak geen nieuw persoonlijk betaalaccount aan.

Iedereen in het team kan de volledige serverlijst zien. Werk daarom uitsluitend aan jouw eigen Droplet en blijf van die van andere gebruikers af. Zie [DigitalOcean: teamlidmaatschap](https://docs.digitalocean.com/platform/teams/how-to/manage-membership/).

</div>

## 4. Eén eigen Droplet aanmaken

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op de DigitalOcean-website

Klik op **Create → Droplet** en loop de onderdelen hieronder van boven naar beneden door. De server wordt pas echt aangemaakt als je onderaan op de laatste knop klikt, dus je kunt alles rustig controleren. Neem uit de schermafbeeldingen alleen de locatie van de knoppen over, niet de voorbeeldwaarden.

</div>

### 4.1 Regio en netwerk binnen het datacenter

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Kies: Amsterdam** (als het goedkoopste plan daar beschikbaar is). Een datacenter in de buurt van België houdt de verbinding snel. Behoud bij **VPC network** gewoon het standaardnetwerk.

</div>

![De regiokeuzelijst en het standaard VPC-netwerk. Het bronvoorbeeld staat op New York.](../afbeeldingen/schermafbeeldingen/03_regio.png)

*Figuur 1. Regio en standaard VPC-netwerk in DigitalOcean (kies zelf Amsterdam in plaats van New York). [Bron](../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

### 4.2 Besturingssysteem

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Kies: OS → Debian → 13 x64.** Zo werkt iedereen met exact hetzelfde besturingssysteem en dezelfde opdrachten. Klik dus op **Debian** en controleer rechts versie **13** (`x64`). Laat de tabbladen *Solutions* en *Custom Images* ongemoeid.

</div>

![De beschikbare besturingssystemen. Klik op Debian 13, Ubuntu is de selectie in het bronvoorbeeld.](../afbeeldingen/schermafbeeldingen/03_besturingssysteem.png)

*Figuur 2. Besturingssysteem kiezen in DigitalOcean (klik zelf op Debian 13). [Bron](../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

### 4.3 Het goedkoopste serverplan

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Kies: Bundled → Basic → Regular → het goedkoopste plan.**

1. Staat er een keuze tussen **Bundled** en **v5**, kies dan **Bundled**.
2. Kies **Basic** en bij CPU Options **Regular**.
3. Klik zo nodig op **Show all plans** en selecteer de goedkoopste optie (**$4 per maand** voor **512 MiB RAM, 1 vCPU en 10 GiB SSD**). Zie je dat bedrag niet meteen, controleer dan of **Regular** en **Show all plans** gekozen zijn, en kies geen duurder plan.

</div>

| Aanduiding | Wat betekent dit? |
|---|---|
| 1 vCPU | De virtuele processor die opdrachten uitvoert. |
| 512 MiB RAM | Het werkgeheugen voor draaiende programma's. |
| 10 GiB SSD | De schijfruimte voor Debian, software en databestanden. |
| 500 GiB transfer | Het inbegrepen uitgaande dataverkeer per maand. |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

De vermelde prijs is de basisprijs exclusief eventuele btw (met vier servers samen $16 per maand). Zie ook de [actuele prijzen](https://www.digitalocean.com/pricing/droplets) en [uitleg over serverplannen](https://docs.digitalocean.com/products/droplets/concepts/choosing-a-plan/).

</div>

![De knoppen Basic, Regular en Show all plans. Het bronvoorbeeld toont Premium Intel en een duurder plan, selecteer zelf Regular en de laagste prijs.](../afbeeldingen/schermafbeeldingen/03_serverplan.png)

*Figuur 3. Serverplan kiezen in DigitalOcean (kies zelf Basic, Regular en de laagste prijs). [Bron](../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

### 4.4 Extra opslag en back-ups

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Additional Storage / Add Volume: uit laten.** Je serverplan heeft al een eigen schijf van 10 GiB. Heb je per ongeluk het extra vak hieronder opengeklapt, haal dan het vinkje bij **Add additional storage** weg.

</div>

![Het uitgeklapte vak voor extra opslag. Laat Add additional storage voor dit practicum uitgeschakeld.](../afbeeldingen/schermafbeeldingen/03_extra_opslag.png)

*Figuur 4. Extra opslag in DigitalOcean (laat Add additional storage uitgeschakeld). [Bron](../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Enable automated backups: uit laten.** Deze betaalde optie maakt automatische serverkopieën, maar wij bewaren onze code en resultaten zelf via GitHub en de notebooks. Zo betaal je alleen voor het basisplan. Zie [DigitalOcean: back-ups](https://docs.digitalocean.com/products/backups/).

</div>

### 4.5 Aanmelden met een wachtwoord

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Kies: Password.** Hiermee stel je het eerste beheerderswachtwoord (`root`) van de server in. Je hoeft dus geen SSH-sleutel aan te maken of te uploaden.

1. Klik onder **Authentication** op **Password**.
2. Kies een sterk, uniek wachtwoord (bij voorkeur uit een wachtwoordmanager) en sla dit op als **`Droplet root-wachtwoord`** (samen met je servernaam). Zet dit wachtwoord nooit in een notebook of op GitHub.

Straks maak je vanuit de terminal met het commando `ssh` een versleutelde verbinding en vul je dat wachtwoord in. Ook als je later een nieuwe Codespace opent, blijft je serveraccount met dit wachtwoord gewoon bestaan.

</div>

![De tabbladen SSH Keys en Password. Klik op Password, het bronvoorbeeld toont nog het andere tabblad.](../afbeeldingen/schermafbeeldingen/03_aanmeldmethode.png)

*Figuur 5. Aanmeldmethode kiezen in DigitalOcean (klik zelf op Password). [Bron](../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Meer informatie: [DigitalOcean: aanmelden via SSH met een wachtwoord of sleutel](https://docs.digitalocean.com/products/droplets/how-to/connect-with-ssh/).

</div>

### 4.6 Netwerk, monitoring en startup scripts

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Stel onder **Additional Options** (klap zo nodig open) alleen het volgende in:

</div>

| Optie | Jouw keuze | Waarom? |
|---|---|---|
| Public IPv4 address | **Aan** | Het internetadres waarmee je vanuit Codespaces verbindt. |
| Public IPv6 address | **Uit** | In dit practicum gebruiken we alleen IPv4. |
| Improved Metrics and monitoring | **Uit** | Niet nodig en houdt de kleine server licht. |
| Startup scripts | **Uit, tekstvak leeg** | We voeren de installatie straks stap voor stap zelf uit. |
| Managed Database | **Uit** | We gebruiken SQLite op de schijf van de Droplet zelf. |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Zie ook [DigitalOcean: uitleg over startup scripts](https://docs.digitalocean.com/products/droplets/how-to/provide-user-data/).

</div>

![Netwerk en aanvullende opties. Houd alleen Public IPv4 aan en zet ook de in dit bronvoorbeeld aangevinkte monitoring uit.](../afbeeldingen/schermafbeeldingen/03_netwerk_extra_opties.png)

*Figuur 6. Netwerk en aanvullende opties in DigitalOcean (houd alleen Public IPv4 aan en zet monitoring uit). [Bron](../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

### 4.7 Aantal, servernaam, project en jouw voornaam als tag

| Optie | Wat vul je in? | Waarom? |
|---|---|---|
| Quantity | **1** | Je maakt één server voor jezelf. |
| Hostname / Give your Droplet a name | De servernaam uit cel 4, bijvoorbeeld `iot-jan-janssens` | Zo herken je jouw server in de lijst en terminal. |
| Select a project | **starten-met-iot** | Houdt alle practicumservers samen in het teamproject. |
| Tags | **Jouw voornaam**, bijvoorbeeld `jan` | Makkelijk om ieders server terug te vinden en te filteren. |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Klik bij **Tags** op **Enter tag name**, typ je voornaam in kleine letters (bij een dubbele voornaam met koppelteken, bijvoorbeeld `jan-pieter`) en druk op Enter. Controleer dat je nog steeds in team **lanupro-tech** werkt. [Uitleg over tags](https://docs.digitalocean.com/products/droplets/how-to/tag/).

</div>

![Aantal, naam, project en tags. Vul één server, starten-met-iot en je eigen voornaam in. De voorbeeldwaarden zijn geen practicuminstellingen.](../afbeeldingen/schermafbeeldingen/03_naam_project_tags.png)

*Figuur 7. Aantal, servernaam, project en tags in DigitalOcean (vul je eigen servernaam en voornaam in). [Bron](../afbeeldingen/schermafbeeldingen/BRONNEN.md).*

### 4.8 Laatste controle en aanmaken

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Loop vóór je klikt deze korte controle na:

- **lanupro-tech → starten-met-iot**
- **Debian 13 x64**
- **Basic → Regular → laagste prijs**, momenteel $4 per maand
- **Password**, met het root-wachtwoord veilig bewaard
- publiek **IPv4 aan**
- extra opslag, automatische back-ups, monitoring, startup scripts en managed database **uit**
- **één** server, met jouw servernaam en **voornaam als tag**.

Controleer het totaalbedrag. Een hoger bedrag dan het gekozen basisplan wijst mogelijk op een extra optie. Klik daarna één keer op **Create Droplet**. Wacht tot de aanmaak gereed is, open jouw server en kopieer het **Public IPv4**-adres. Dat vul je hieronder in bij de variabele `mijn_server_ip`.

</div>

<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Kosten</strong><br>Een Droplet wordt niet automatisch na een maand verwijderd. Ook een uitgeschakelde Droplet kan kosten blijven veroorzaken doordat de resources gereserveerd blijven. De docent beheert de einddatum en verwijdert de oefenservers na het veilig bewaren van de benodigde resultaten. Controleer de actuele tarieven via <a href="https://docs.digitalocean.com/products/droplets/details/pricing/">DigitalOcean</a>.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 03)

Vul in de codecel hieronder tussen de aanhalingstekens het **Public IPv4**-adres van jouw eigen Droplet in bij `mijn_server_ip` en voer die cel uit. Let op: op het overzicht van je Droplet staat soms ook een **Private IP** (begint meestal met `10.`). Gebruik altijd het **Public IPv4**-adres, want het interne adres werkt niet vanaf Codespaces.

Voer daarna ook de cel eronder uit: die controleert of het adres geldig is, bewaart het lokaal in `gegevens/werk/server.json` (buiten Git) en maakt meteen je eerste SSH-opdracht.

*(Werk je op een andere dag verder in dezelfde Codespace? Dan mag je `mijn_server_ip` leeg laten, want het eerder opgeslagen adres uit `server.json` wordt automatisch opnieuw gebruikt.)*

</div>

In [3]:
# Vul hieronder tussen de aanhalingstekens het publieke IPv4-adres van jouw Droplet in:
mijn_server_ip = "178.62.241.168"  

In [4]:
serverpad = werkmap / "server.json"
invoer_ip = globals().get("mijn_server_ip", "").strip()
if invoer_ip:
    serveradres = str(ipaddress.IPv4Address(invoer_ip))
    serverinstellingen = {"ipv4": serveradres, "deelnemerscode": deelnemer}
    serverpad.write_text(json.dumps(serverinstellingen, indent=2) + "\n", encoding="utf-8")
elif serverpad.exists():
    serverinstellingen = json.loads(serverpad.read_text(encoding="utf-8"))
    if serverinstellingen.get("deelnemerscode") != deelnemer:
        raise ValueError("Dit serverbestand hoort bij een andere deelnemer. Controleer je werkmap.")
    serveradres = str(ipaddress.IPv4Address(serverinstellingen["ipv4"]))
else:
    raise ValueError("Vul eerst het publieke IPv4-adres van jouw Droplet in bij 'mijn_server_ip' in de cel hierboven.")
print("Controleer dit adres met jouw Droplet:", serveradres)

Controleer dit adres met jouw Droplet: 178.62.241.168


## 5. Je eerste beheerverbinding

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Nu je Droplet draait, maak je vanuit Codespaces verbinding met die server zodat je daar straks software kunt installeren en bestanden kunt overzetten. Daarvoor gebruik je **SSH** (*Secure Shell*): een versleutelde tekstverbinding waarmee je opdrachten veilig over internet naar je server stuurt (niemand onderweg kan je wachtwoord of commando's meelezen).

Omdat je nog nooit eerder met deze nieuwe server hebt verbonden, wil SSH eerst zeker weten dat je ook echt met jóuw server praat en niet met een valse tussenserver. Daarom vraag je eerst in de **Web Console** op DigitalOcean de unieke **SHA256-vingerafdruk** van de publieke serversleutel op, zodat je die straks kunt vergelijken.

**Uitvoeren:** op de DigitalOcean-website (Web Console van jouw Droplet)

Open je Droplet op DigitalOcean, klik op **Web Console** en voer onderstaande opdrachten uit:

</div>

```bash
whoami
hostname
ssh-keygen -lf /etc/ssh/ssh_host_ed25519_key.pub
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Met `ssh-keygen -lf` vraag je de vingerafdruk van de **publieke** serversleutel (`.pub`) op. De bijbehorende privésleutel (het bestand zonder `.pub`) blijft altijd geheim op de server. Je verwacht `root`, jouw servernaam en een regel met `SHA256:...`. Noteer of bewaar dat scherm om dadelijk te vergelijken.

</div>

![Serveroverzicht met Web Console](../afbeeldingen/schermafbeeldingen/03_serveroverzicht.png)

*Figuur 8. Serveroverzicht uit de officiële DigitalOcean-documentatie. De afbeelding toont de knop Web Console en het veld Public IPv4. Naam, adres, besturingssysteem en prijs in dit voorbeeld horen niet bij jouw server. [Bron](https://docs.digitalocean.com/products/droplets/how-to/connect-with-console/).*

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 03 en Terminal)

Voer nu de onderstaande Python-cel uit. Die zet jouw eerder ingevulde IP-adres in het `ssh`-commando en drukt de volledige opdracht voor de Codespaces-terminal af:

</div>

In [27]:
# Codespaces (Linux bash)
print(f"ssh -o HostKeyAlgorithms=ssh-ed25519 root@{serveradres}")

ssh -o HostKeyAlgorithms=ssh-ed25519 root@178.62.241.168


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

1. Open in Codespaces een terminal via **Terminal → New Terminal**.
2. Plak de afgedrukte `ssh`-opdracht erin en druk op Enter.
3. Vergelijk de getoonde `SHA256:...`-vingerafdruk met wat je in de Web Console zag. Komen ze overeen, typ dan voluit `yes` en druk op Enter.
4. Vul daarna jouw **`Droplet root-wachtwoord`** in (uit Stap 4.5). *(Tijdens het typen zie je geen letters of sterretjes, dat is normaal.)*

Zodra je bent ingelogd (`root@...:~#`), stuurt deze terminal je opdrachten rechtstreeks naar **jouw Debian-server** (niet meer naar Codespaces). Voer daar ter controle deze twee opdrachten uit (je verwacht dezelfde gebruiker en servernaam als in de Web Console, en met `exit` keer je later terug):

</div>

```bash
whoami
hostname
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

#### Uitzondering: werk je lokaal op je eigen Windows- of Mac-computer (niet in Codespaces)?

De standaardroute hierboven gaat uit van de terminal in Codespaces (Linux). Als je dit notebook toch lokaal op je eigen laptop uitvoert, kun je hieronder de `ssh`-opdracht afdrukken voor **macOS** (Terminal) of **Windows** (PowerShell of Git Bash). De twee controle-opdrachten (`whoami` en `hostname`) hierboven voer je dan pas uit *nadat* je via `ssh` op je Debian-server bent ingelogd.

</div>

In [5]:
# Alleen als je lokaal op macOS werkt (Terminal)
print(f"ssh -o HostKeyAlgorithms=ssh-ed25519 root@{serveradres}")

ssh -o HostKeyAlgorithms=ssh-ed25519 root@178.62.241.168


In [19]:
# Alleen als je lokaal op Windows werkt (PowerShell of Git Bash)
print(f"ssh -o HostKeyAlgorithms=ssh-ed25519 root@{serveradres}")

ssh -o HostKeyAlgorithms=ssh-ed25519 root@64.225.71.10


<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Tip: als aanmelden niet lukt</strong><br>Gebruik de Web Console om gebruikers of instellingen na te kijken. Als SSH of de netwerkconfiguratie zelf defect is, gebruik dan de <a href="https://docs.digitalocean.com/products/droplets/how-to/recovery/recovery-console/">Recovery Console</a> via DigitalOcean. Die is een afzonderlijke herstelroute en vereist een serverwachtwoord. Verwijder een waarschuwing over een gewijzigde serversleutel niet zonder eerst opnieuw de vingerafdruk te controleren.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 2.** Waarom gebruiken we voor het beheer van de server een **SSH-verbinding**, en waarom controleer je bij de eerste keer aanmelden eerst de `SHA256`-vingerafdruk voordat je jouw wachtwoord intypt?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 2):*het is een veilige versleutelde manier om in te loggen op het netwerk en op je server*

*Voeg hier je antwoord in. Tip: leg in twee korte zinnen uit (1) wat SSH doet met je verbinding en (2) wat je voorkomt door de vingerafdruk vooraf te controleren.*

</div>

## 6. Een gebruiker voor dagelijks beheer maken

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als root)

`root` mag alles op de server aanpassen. Voor dagelijks beheer is dat onveilig, dus maak je een aparte gebruiker **beheer** aan. Met `sudo` kan dat account wel gericht één opdracht met beheerrechten uitvoeren.

Werk eerst de lijst met beschikbare pakketten bij, installeer beveiligingsupdates en voeg de gebruiker toe. Voer ieder blok afzonderlijk uit en controleer dat het zonder fout eindigt.

</div>

```bash
apt-get update
apt-get upgrade -y
apt-get install -y sudo
```

```bash
adduser beheer
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

`adduser beheer` stelt je nu een paar vragen in de terminal:
1. **Nieuw wachtwoord:** kies een **nieuw**, sterk wachtwoord speciaal voor `beheer` (dus *niet* het `Droplet root-wachtwoord`), sla dit in je wachtwoordmanager op als **`SSH beheer-wachtwoord`** en typ het twee keer in *(tijdens het typen zie je geen sterretjes)*.
2. **Gebruikersgegevens (`Full Name`, `Room Number`, `Work Phone`, `Home Phone`, `Other`):** vul eventueel alleen je naam in bij `Full Name` en druk voor de rest steeds gewoon op **Enter** om de velden leeg te laten.
3. **Bevestigen (`Is the information correct? [Y/n]`):** typ `y` en druk op **Enter**.

Pas nadat `adduser beheer` volledig is afgerond, geef je dit account beheerrechten (`sudo`) en controleer je het resultaat met de volgende opdracht *(bestaat `beheer` al omdat je later verderwerkt, sla `adduser beheer` dan over en voer alleen onderstaande twee regels uit)*:

</div>

```bash
usermod -aG sudo beheer
id beheer
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Bij `id beheer` moet nu in de groepenlijst **`sudo`** staan.

**Uitvoeren:** in Codespaces (een tweede Terminal)

Laat de eerste SSH-terminal open. Open met **Terminal → New Terminal** een tweede terminal in Codespaces en voer de cel hieronder uit om je `ssh`-opdracht voor gebruiker `beheer` af te drukken:

</div>

In [6]:
# Codespaces (Linux bash)
print(f"ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}")

ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@178.62.241.168


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

*(Werk je toch lokaal op macOS of Windows in plaats van in Codespaces? Gebruik dan één van de twee cellen hieronder:)*

</div>

In [28]:
# Alleen lokaal op macOS (Terminal)
print(f"ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}")

ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@178.62.246.100


In [22]:
# Alleen lokaal op Windows (PowerShell of Git Bash)
print(f"ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}")

ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@64.225.71.10


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Kopieer de `ssh`-opdracht naar de tweede terminal en meld je aan met jouw **`SSH beheer-wachtwoord`** (uit Stap 6).

Het terminalvenster zelf draait in Codespaces (of eventueel lokaal op je eigen computer, zoals in het Windows PowerShell-voorbeeld hieronder), maar via **`ssh`** staat het nu in directe verbinding met jouw virtuele machine (Droplet) bij DigitalOcean. Zodra je onderaan een prompt ziet die begint met `beheer@...:~$`, heb je dus een **SSH-terminal op die server**, aangemeld als gebruiker **`beheer`**.

</div>

![Aangemeld als gebruiker beheer via SSH op de Droplet](../afbeeldingen/schermafbeeldingen/03_ssh_beheer.png)

*Figuur 9. Voorbeeld van een directe SSH-verbinding als gebruiker `beheer` (hier vanuit Windows PowerShell). Na het inloggen verschijnt de serverprompt `beheer@...:~$`.*

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je DigitalOcean-server (deze nieuwe SSH-terminal, als `beheer`)

Controleer nu op die server zelf je gebruikersnaam en of `sudo` werkt:

</div>

```bash
whoami
sudo -v
sudo whoami
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

De eerste opdracht geeft `beheer`, en de laatste geeft `root`. `sudo -v` vraagt zo nodig opnieuw het wachtwoord van **beheer**. Ga pas verder wanneer beide aanmeldingen werken. Sluit dan de oude root-terminal met `exit`.

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Maak direct aanmelden als root via SSH onmogelijk, terwijl wachtwoordaanmelding voor beheer beschikbaar blijft. `tee` schrijft de regels tussen de twee `EOF`-markeringen naar een configuratiebestand.

</div>

```bash
sudo tee /etc/ssh/sshd_config.d/00-practicum.conf >/dev/null <<'EOF'
PasswordAuthentication yes
PermitRootLogin no
MaxAuthTries 3
EOF
sudo /usr/sbin/sshd -t
sudo /usr/sbin/sshd -T | grep -E '^(passwordauthentication|permitrootlogin|maxauthtries) '
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Je verwacht `passwordauthentication yes`, `permitrootlogin no` en `maxauthtries 3`. Als de controle afwijkt of een fout geeft, herstel het bestand vóór je herlaadt. Bij een correcte controle:

</div>

```bash
sudo systemctl reload ssh
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Houd je huidige verbinding open en meld je in een nieuwe terminal nogmaals aan als **beheer**. Zo controleer je dat de nieuwe instellingen werken voordat je een werkende sessie sluit.

</div>

## 7. De programma's installeren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Op een lege Debian-server staat bijna niets voorgeïnstalleerd. Met `apt-get` (de pakketbeheerder van Debian) installeer je nu in één keer alle software die we nodig hebben: **Mosquitto** (de MQTT-broker), **Python**, **SQLite**, **Git**, **Nginx** (voor de HTTPS-webverbinding), **OpenSSL** (voor certificaten) en **UFW** (de firewall).

</div>

```bash
sudo apt-get install -y mosquitto mosquitto-clients python3 python3-venv sqlite3 git openssl ca-certificates nginx ufw
sudo systemctl disable --now mosquitto
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Controleer de geïnstalleerde software en het besturingssysteem:

</div>

```bash
cat /etc/os-release
python3 --version
mosquitto -h | head -n 2
sqlite3 --version
git --version
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Je verwacht Debian 13, Python 3.13 en Mosquitto 2.0.x (het laatste cijfer van het versienummer kan door beveiligingsupdates licht afwijken). Overschrijf de systeem-Python verder niet handmatig met `pip`.

Een **proces** is een programma dat op dit moment draait, en een **service** laat je met `systemctl` starten en stoppen. Omdat Debian Mosquitto bij het installeren soms meteen opstart, zetten we hem hieronder eerst bewust uit met `disable --now`. Je start de broker pas zodra je de certificaten en wachtwoorden hebt ingesteld. Controleer nu eerst de status:

</div>

```bash
sudo systemctl status mosquitto --no-pager
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Je verwacht ongeveer onderstaande uitvoer. Het lege bolletje (`○`), `disabled` en `inactive (dead)` zien er misschien vreemd uit als je nog nooit met `systemctl` gewerkt hebt, maar betekenen hier precies dat alles gelukt is:

</div>

```text
○ mosquitto.service - Mosquitto MQTT Broker
     Loaded: loaded (/usr/lib/systemd/system/mosquitto.service; disabled; preset: enabled)
     Active: inactive (dead)
...
Stopping mosquitto.service - Mosquitto MQTT Broker...
Stopped mosquitto.service - Mosquitto MQTT Broker.
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- **`Loaded: ... disabled`** betekent dat Mosquitto niet vanzelf opstart bij een serverherstart (zoals `disable --now` heeft ingesteld).
- **`Active: inactive (dead)`** en **`Stopped mosquitto.service`** onderaan betekenen dat de broker nu bewust gestopt is. Dat is precies de bedoeling: Mosquitto is al geïnstalleerd, maar je start hem pas zodra je in de volgende stappen de certificaten en wachtwoorden hebt ingesteld.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 3.** Kijk naar de uitvoer van `python3 --version`, `mosquitto -h | head -n 2` en `sudo systemctl status mosquitto --no-pager` op jouw server:
1. Welke versienummers van Python en Mosquitto staan er geïnstalleerd?
2. Waarom laten we Mosquitto nu bewust op `inactive (dead)` en `disabled` staan in plaats van hem al meteen te laten draaien?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 3):*Python 3.13.5 en mosquitto 2.47.3, we passen de configuratie aan en maken dat hij niet ongewenst dingen doet*

*Voeg hier je antwoord in. Tip: noteer (1) de versienummers uit jouw eigen terminal en (2) waarom we nog wachten met starten (en wat `disabled` betekent na een serverherstart).*

</div>

## 8. De meegeleverde serverbestanden overbrengen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Terminal, zonder SSH-verbinding)

In de map `server/` staan de programma's en configuratiebestanden voor jouw server. Met **SCP** kopieer je die map via dezelfde versleutelde verbinding naar de Droplet.

Open een nieuwe terminal (zonder SSH-verbinding) in Codespaces en ga naar de hoofdmap van je repository om te controleren of de map `server/` aanwezig is:

</div>

```bash
cd "$(git rev-parse --show-toplevel)"
ls server
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Je verwacht onder meer `certificaat_maken.py`, `toegang_api.py`, `statusdienst.py`, `requirements.txt`, `configuratie/` en `diensten/`. Ontbreekt de map, dan heeft je branch de bestanden van notebook 03 nog niet. Bewaar je werk en laat je branch bijwerken voordat je doorgaat.

Voer nu de cel hieronder uit om het `scp`-commando voor Codespaces af te drukken, plak dat in de Codespaces-terminal en vul jouw **`SSH beheer-wachtwoord`** in:

</div>

In [7]:
# Codespaces (Linux bash)
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 -r server beheer@{serveradres}:~/practicum-server")

scp -o HostKeyAlgorithms=ssh-ed25519 -r server beheer@178.62.241.168:~/practicum-server


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

#### Uitzondering: werk je lokaal op macOS of Windows (niet in Codespaces)?

Gebruik dan eerst het commando voor jouw systeem om naar de projectmap te gaan en kopieer daarna de map met een van de onderstaande `scp`-cellen:

</div>

```bash
# Alleen lokaal op macOS (Terminal)
cd "$(git rev-parse --show-toplevel)"
ls server
```

```powershell
# Alleen lokaal op Windows (PowerShell)
cd (git rev-parse --show-toplevel)
dir server
```

In [8]:
# Alleen lokaal op macOS (Terminal)
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 -r server beheer@{serveradres}:~/practicum-server")

scp -o HostKeyAlgorithms=ssh-ed25519 -r server beheer@64.225.71.10:~/practicum-server


In [10]:
# Alleen lokaal op Windows (PowerShell)
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 -r server beheer@{serveradres}:~/practicum-server")

scp -o HostKeyAlgorithms=ssh-ed25519 -r server beheer@64.225.71.10:~/practicum-server


<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Tip: als je de bestanden later nog een keer wilt kopiëren</strong><br>Voer bovenstaande <code>scp</code>-opdracht maar één keer uit. Als de map <code>~/practicum-server</code> al op je server staat en je voert hetzelfde commando nogmaals uit, dan maakt <code>scp</code> per ongeluk een dubbele submap aan (<code>~/practicum-server/server/</code>) in plaats van de bestaande bestanden bij te werken. Moet je de bestanden op een later moment echt opnieuw overschrijven, voeg dan <code>/.</code> achter <code>server</code> toe en een <code>/</code> aan het einde: <code>scp -o HostKeyAlgorithms=ssh-ed25519 -r server/. beheer@JOUW_IP:~/practicum-server/</code>.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Controleer dat de bestanden op de server aangekomen zijn:

</div>

```bash
ls ~/practicum-server
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Nu richt je op de server een veilige, aparte werkplek in voor onze webdienst (die later de API gaat aanbieden). Daarvoor maak je eerst een apart **systeemaccount `iot`** aan.

**Waarom een apart systeemaccount `iot`?** Een webdienst is straks vanaf internet bereikbaar. Zo'n programma wil je uit veiligheid nooit laten draaien onder `root` of onder jouw beheeraccount `beheer`, want bij een fout of veiligheidslek zou iemand dan meteen systeeminstellingen kunnen aanpassen. Het systeemaccount `iot` krijgt geen wachtwoord, kan niet via SSH inloggen (`nologin`) en heeft geen `sudo`-rechten: het mag alleen onze eigen webdienst draaien en bijbehorende bestanden bewaren.

Voer het blok hieronder uit op je server. Dit doen de zes regels achter elkaar:
1. **Regel 1 (`useradd ... iot`):** maakt het systeemaccount `iot` aan (als het nog niet bestaat) met de map `/var/lib/iot` voor latere servergegevens.
2. **Regel 2 (`install -d ...`):** maakt de vaste mappen `/opt/iot/programmatuur` (voor onze Python-code) en `/etc/iot` (voor instellingen) aan.
3. **Regel 3 (`python3 -m venv`):** maakt in `/opt/iot/venv` een **virtuele Python-omgeving** aan, een afgeschermde Python-map zodat onze pakketten de systeem-Python van Debian niet verstoren.
4. **Regel 4 (`pip install`):** installeert in die virtuele omgeving de benodigde pakketten uit `requirements.txt`, waaronder **FastAPI** (om web-API's te bouwen) en **Uvicorn** (de webserver die het FastAPI-programma straks draait).
5. **Regel 5 (`install -m 644`):** kopieert de twee meegeleverde Python-bestanden (`statusdienst.py` en `toegang_api.py`) naar `/opt/iot/programmatuur/`.
6. **Regel 6 (`pip check`):** controleert of alle geïnstalleerde pakketten in de virtuele omgeving goed bij elkaar passen.

</div>

```bash
id iot >/dev/null 2>&1 || sudo useradd --system --user-group --home-dir /var/lib/iot --create-home --shell /usr/sbin/nologin iot
sudo install -d -m 755 /opt/iot/programmatuur /etc/iot
sudo python3 -m venv /opt/iot/venv
sudo /opt/iot/venv/bin/python -m pip install -r ~/practicum-server/requirements.txt
sudo install -m 644 ~/practicum-server/statusdienst.py ~/practicum-server/toegang_api.py /opt/iot/programmatuur/
sudo /opt/iot/venv/bin/python -m pip check
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Verwachte uitvoer:** het laatste commando (`pip check`) eindigt met **`No broken requirements found.`**. Daarmee zijn het systeemaccount `iot` en de aparte Python-omgeving klaar voor de beveiligde testdienst die we in Stap 11 gaan aanzetten.

</div>

## 9. Versleuteling en de identiteit van je server

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Zonder versleuteling zouden jouw wachtwoorden en meetgegevens leesbaar over het internet reizen. Daarom beveiligen we zowel MQTT (poort 8883) als de webverbinding (HTTPS, poort 443) met **TLS-certificaten**.

Omdat we het vaste IPv4-adres van je Droplet gebruiken zonder een eigen domeinnaam te kopen, maak je met onderstaand script zelf een kleine **certificaatautoriteit (CA)** aan die het servercertificaat ondertekent. Hoe werken deze vier bestanden straks samen wanneer een client (zoals Codespaces) verbindt?

- **`ca.crt` is openbare controle-informatie:** je mag dit bestand veilig naar Codespaces (en straks naar een andere gebruiker) kopiëren. De client gebruikt `ca.crt` om te controleren of het certificaat van de server echt door jouw CA ondertekend is. Wie alleen `ca.crt` heeft, kan zich **niet** voordoen als jouw server.
- **`server.crt` is de identiteitskaart van de server:** de server stuurt dit certificaat bij elke verbinding mee. Daarin staat ook het **IPv4-adres** waarvoor dit certificaat geldig is (`IP Address`). Door dat IP-adres mee te controleren, weet je zeker dat het certificaat niet stiekem van een *andere* server is gekopieerd.
- **`server.key` en `ca.key` zijn geheime privésleutels:** alleen wie `server.key` heeft, kan bewijzen dat hij echt de eigenaar van `server.crt` is en mag het verkeer ontsleutelen. Als een privésleutel uitlekt (bijvoorbeeld op GitHub), kan iemand zich voordoen als jouw server. Daarom blijven `server.key` en `ca.key` altijd op de server zelf.

</div>

| Bestand | Betekenis | Waar blijft het? |
|---|---|---|
| `ca.crt` | Openbaar certificaat waarmee je de ondertekening controleert | Server en Codespaces, en later ook bij een andere gebruiker |
| `ca.key` | Privésleutel waarmee de CA ondertekent | Alleen op de server, toegankelijk voor root |
| `server.crt` | Certificaat met het IPv4-adres van deze server | Op de server. De server stuurt het bij de verbinding mee |
| `server.key` | Privésleutel van de server | Alleen op de server, met beperkte leesrechten |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 03) en op je DigitalOcean-server (SSH-terminal, als beheer)

Voer de Python-cel hieronder uit in Notebook 03. Die vult jouw serveradres en deelnemerscode meteen in. Kopieer de afgedrukte opdrachten naar je SSH-terminal (`beheer`) om het certificaat aan te maken en te controleren:

</div>

In [8]:
print(f"sudo python3 ~/practicum-server/certificaat_maken.py --adres {serveradres} --deelnemer {deelnemer}")
print("sudo openssl x509 -in /etc/iot/certificaten/server.crt -noout -dates -ext subjectAltName")

sudo python3 ~/practicum-server/certificaat_maken.py --adres 178.62.241.168 --deelnemer kenzo_meulemeester
sudo openssl x509 -in /etc/iot/certificaten/server.crt -noout -dates -ext subjectAltName


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Controleer dat `IP Address:` jouw publieke IPv4-adres noemt. Het servercertificaat is 90 dagen geldig. Bestaat het certificaat al voor dit IP-adres, dan blijft het gewoon behouden.

Maak nu de openbare CA beschikbaar om die via SCP op te halen en geef Mosquitto zijn eigen kopie van het servercertificaat en de privésleutel (alleen `root` en Mosquitto mogen die sleutel lezen). Voer de cel hieronder uit en plak de vier afgedrukte opdrachten in je SSH-terminal:

</div>

In [9]:
print("sudo install -m 644 /etc/iot/certificaten/ca.crt /etc/iot/ca.crt")
print("sudo install -o root -g mosquitto -m 640 /etc/iot/certificaten/server.crt /etc/mosquitto/certs/server.crt")
print("sudo install -o root -g mosquitto -m 640 /etc/iot/certificaten/server.key /etc/mosquitto/certs/server.key")
print(f"sudo openssl verify -CAfile /etc/iot/ca.crt -verify_ip {serveradres} /etc/iot/certificaten/server.crt")

sudo install -m 644 /etc/iot/certificaten/ca.crt /etc/iot/ca.crt
sudo install -o root -g mosquitto -m 640 /etc/iot/certificaten/server.crt /etc/mosquitto/certs/server.crt
sudo install -o root -g mosquitto -m 640 /etc/iot/certificaten/server.key /etc/mosquitto/certs/server.key
sudo openssl verify -CAfile /etc/iot/ca.crt -verify_ip 178.62.241.168 /etc/iot/certificaten/server.crt


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

De laatste regel hoort te eindigen op `OK`. Bewaar de privésleutels nooit in je repository. Als je later de Droplet vervangt en een ander IP-adres krijgt, moet de certificaatinrichting opnieuw gecontroleerd worden.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 4.** Waarom kun je `ca.crt` wel veilig naar Codespaces kopiëren, terwijl `server.key` altijd op de server moet blijven? En waarom controleer je ook of jouw IP-adres in `server.crt` staat?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 4):**

*Het eerste is een openbaar certificaat dat iedereen kan zien, terwijl het 2de een beveiligde sleutel is die strikt geheim moet blijven. Je client zal controleren als of het IP adres in het certificaat staat/ Tip: leg (1) het verschil uit tussen het openbare controlebestand `ca.crt` en de geheime privésleutel `server.key`, en (2) wat de controle van het IP-adres in `server.crt` voorkomt.*

</div>

## 10. De MQTT-broker toegankelijk maken

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Nu je broker straks via internet bereikbaar is, mag niet iedereen zomaar berichten lezen of plaatsen. Daarom stel je op de Mosquitto-broker twee aparte **MQTT-clientaccounts** in, elk met een **nieuw, zelfgekozen sterk wachtwoord** (andere wachtwoorden dan voor SSH):
- **`eigenaar`:** de login voor **jouw eigen MQTT-clients** (je Python-scripts die als *publisher* metingen versturen én als *subscriber* berichten ophalen op jouw broker). Sla dit wachtwoord in je wachtwoordmanager bijvoorbeeld op als **`MQTT-broker eigenaar (schrijven + lezen)`**.
- **`lezer`:** de login voor de **MQTT-client van een andere gebruiker**, zodat die als *subscriber* alleen jouw meetberichten kan meelezen zonder iets op jouw broker te kunnen schrijven. Sla dit bijvoorbeeld op als **`MQTT-broker lezer (alleen lezen)`**.

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Voer de onderstaande stappen **één voor één achter elkaar** uit in je SSH-terminal:

**Stap 1:** Wachtwoordbestand aanmaken en een nieuw wachtwoord voor `eigenaar` instellen  
De eerste regel maakt een leeg wachtwoordbestand aan (`/etc/mosquitto/practicum.wachtwoorden`, als dat nog niet bestaat). Daarna vraagt `mosquitto_passwd` twee keer om jouw **nieuwe** wachtwoord voor `eigenaar` (tijdens het typen zie je geen sterretjes). *(Zie je eerst `Warning: File ... group is not root` vlak vóór `Password:`? Dat is normaal, want de groep staat op `mosquitto` zodat de broker het bestand kan lezen.)*

</div>

```bash
sudo test -e /etc/mosquitto/practicum.wachtwoorden || sudo install -o root -g mosquitto -m 640 /dev/null /etc/mosquitto/practicum.wachtwoorden
sudo mosquitto_passwd /etc/mosquitto/practicum.wachtwoorden eigenaar
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Stap 2:** Een nieuw wachtwoord voor `lezer` instellen  
Voeg nu het tweede account (`lezer`) aan datzelfde wachtwoordbestand toe met een eigen, ander sterk wachtwoord en sla dit op als **`MQTT-broker lezer (alleen lezen)`** (ook hier is de eventuele `Warning: ... group is not root` vóór `Password:` normaal):

</div>

```bash
sudo mosquitto_passwd /etc/mosquitto/practicum.wachtwoorden lezer
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Stap 3:** Leesrechten op het wachtwoordbestand beveiligen  
Met `chown` en `chmod 640` zorg je ervoor dat uitsluitend `root` en de Mosquitto-service dit wachtwoordbestand mogen lezen, zodat niemand anders op de server erbij kan:

</div>

```bash
sudo chown root:mosquitto /etc/mosquitto/practicum.wachtwoorden
sudo chmod 640 /etc/mosquitto/practicum.wachtwoorden
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

*(Werk je later verder, dan hoef je de wachtwoorden niet opnieuw in te stellen. Voer je één van de `mosquitto_passwd`-regels nogmaals uit, dan overschrijf je alleen dat wachtwoord.)*

**Stap 4:** Topicrechten instellen (`/etc/mosquitto/practicum.rechten`)  
Een wachtwoord bepaalt *wie* mag aanmelden op je broker, maar nog niet *wat* elk account daarna mag doen. In het rechtenbestand leg je per account vast op welke MQTT-onderwerpen (**topics**) gelezen en geschreven mag worden:
- **`eigenaar`** krijgt **`readwrite`** (lezen én publiceren) op alles onder `practicum/jouw_deelnemerscode/#` (`#` geldt als joker voor alle subtopics eronder).
- **`lezer`** krijgt uitsluitend **`read`** (alleen lezen, niets publiceren) en alleen op de twee subtopics `/metingen/#` en `/controle/#`.

Voer de Python-cel hieronder uit in Codespaces (Notebook 03). Die zet jouw deelnemerscode automatisch in de juiste terminalopdracht. Plak de volledige uitvoer daarvan in je SSH-terminal op de server:

</div>

In [10]:
print(f"""sudo tee /etc/mosquitto/practicum.rechten >/dev/null <<'EOF'
user eigenaar
topic readwrite practicum/{deelnemer}/#

user lezer
topic read practicum/{deelnemer}/metingen/#
topic read practicum/{deelnemer}/controle/#
EOF
sudo chown root:mosquitto /etc/mosquitto/practicum.rechten
sudo chmod 640 /etc/mosquitto/practicum.rechten
sudo cat /etc/mosquitto/practicum.rechten""")

sudo tee /etc/mosquitto/practicum.rechten >/dev/null <<'EOF'
user eigenaar
topic readwrite practicum/kenzo_meulemeester/#

user lezer
topic read practicum/kenzo_meulemeester/metingen/#
topic read practicum/kenzo_meulemeester/controle/#
EOF
sudo chown root:mosquitto /etc/mosquitto/practicum.rechten
sudo chmod 640 /etc/mosquitto/practicum.rechten
sudo cat /etc/mosquitto/practicum.rechten


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Het laatste commando (`sudo cat`) toont de inhoud van het rechtenbestand. Controleer dat jouw deelnemerscode in alle drie de `topic`-regels juist staat (dit bestand bevat geen wachtwoorden).

**Stap 5:** Brokerconfiguratie activeren en Mosquitto starten  
Kopieer nu het meegeleverde configuratiebestand (`mosquitto.conf`, dat poort 8883, jouw TLS-certificaat en de twee controlebestanden hierboven inschakelt) naar Mosquitto, houd automatisch starten uit (`disable`) en start de broker handmatig:

</div>

```bash
sudo install -m 644 ~/practicum-server/configuratie/mosquitto.conf /etc/mosquitto/conf.d/practicum.conf
sudo systemctl disable mosquitto
sudo systemctl start mosquitto
sudo systemctl status mosquitto --no-pager
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Verwachte uitvoer:** je ziet ongeveer het onderstaande resultaat. Een groen bolletje (`●`), **`Active: active (running)`** en de regel **`Loading config file /etc/mosquitto/conf.d/practicum.conf`** bevestigen dat jouw configuratie actief is en dat Mosquitto beveiligd luistert op poort **8883** (terwijl `disabled` bij `Loaded:` ervoor zorgt dat hij na een serverherstart uit blijft totdat je hem zelf start). *(Een eventuele `Note:`-melding over `message_size_limit` is onschuldig.)*

</div>

```text
● mosquitto.service - Mosquitto MQTT Broker
     Loaded: loaded (/usr/lib/systemd/system/mosquitto.service; disabled; preset: enabled)
     Active: active (running)
...
Loading config file /etc/mosquitto/conf.d/practicum.conf
Started mosquitto.service - Mosquitto MQTT Broker.
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Krijg je toch een foutmelding bij het starten, bekijk dan de logregels onderaan bij Hulp bij foutmeldingen.

</div>

## 11. De beveiligde webverbinding klaarzetten

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Om straks vanuit je notebooks via HTTPS meetgegevens van jouw server op te vragen, zetten we nu alvast een kleine beveiligde webdienst (`/gezondheid`) aan. **Nginx** vangt het versleutelde HTTPS-verkeer op poort 443 op en geeft dat binnen de server door aan onze Python-dienst (Uvicorn op `127.0.0.1:8000`, die dus niet rechtstreeks van buitenaf bereikbaar is).

**Stap 1:** Een afzonderlijk API-wachtwoord instellen en beveiligen  
Voor deze HTTPS-API kies je opnieuw een **nieuw, zelfgekozen wachtwoord** (dus *niet* je SSH-wachtwoord en *niet* je MQTT-wachtwoorden). Gebruik minstens **20 willekeurige tekens** zonder accenten of andere speciale niet-ASCII-tekens, en sla het in je wachtwoordmanager op als bijvoorbeeld **`API-Droplet wachtwoord (HTTPS / FastAPI)`**. `toegang_api.py` bewaart alleen een veilige controlewaarde (hash) in `/etc/iot/api.controlewaarde` en drukt het wachtwoord nergens af.

Voer eerst dit commando uit en typ twee keer jouw nieuwe API-wachtwoord in:

</div>

```bash
sudo python3 ~/practicum-server/toegang_api.py
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Zorg daarna dat uitsluitend `root` en het systeemaccount `iot` (dat onze webdienst draait) deze controlewaarde mogen lezen *(als je later hervat en `toegang_api.py` meldt dat de bestaande toegang behouden blijft, gebruik je gewoon je eerder bewaarde **`API-Droplet wachtwoord`**)*:

</div>

```bash
sudo chown root:iot /etc/iot/api.controlewaarde
sudo chmod 640 /etc/iot/api.controlewaarde
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Stap 2:** De Python-statusdienst (`iot-status`) en Nginx-configuratie activeren  
De zeven regels hieronder (1) installeren en starten onze Python-testdienst (`iot-status`) zodat die automatisch blijft draaien, (2) zetten de bijbehorende Nginx-HTTPS-configuratie aan (en halen de standaard Nginx-welkomstpagina weg) en (3) testen met `nginx -t` of de instellingen kloppen:

</div>

```bash
sudo install -m 644 ~/practicum-server/diensten/iot-status.service /etc/systemd/system/iot-status.service
sudo systemctl daemon-reload
sudo systemctl enable --now iot-status
sudo install -m 644 ~/practicum-server/configuratie/nginx.conf /etc/nginx/sites-available/practicum
sudo ln -sfn /etc/nginx/sites-available/practicum /etc/nginx/sites-enabled/practicum
sudo rm -f /etc/nginx/sites-enabled/default
sudo nginx -t
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Je verwacht onderaan `syntax is ok` en `test is successful`. Ga pas verder als die twee regels verschijnen.

**Stap 3:** Nginx herstarten en alle drie de services controleren  
Herstart Nginx zodat de nieuwe HTTPS-configuratie actief wordt, zet automatisch starten bij herstarten aan voor Nginx, en controleer daarna zowel welke services nu draaien (`is-active`) als welke automatisch opstarten (`is-enabled`):

</div>

```bash
sudo systemctl restart nginx
sudo systemctl enable nginx
sudo systemctl is-active mosquitto iot-status nginx
sudo systemctl is-enabled mosquitto iot-status nginx
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Verwachte uitvoer:** je krijgt zes regels onder elkaar: eerst drie keer **`active`** (alle drie de services draaien nu), en daarna **`disabled`**, **`enabled`**, **`enabled`** (want `iot-status` en `nginx` starten vanzelf bij een serverherstart, terwijl we `mosquitto` bewust handmatig starten).

</div>

## 12. Alleen de benodigde poorten openen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Tot slot sluit je met de **firewall (`ufw`)** alle onnodige ingangen van je server af. Je zet alleen de drie poorten open die we echt nodig hebben: **22** (SSH), **8883** (beveiligd MQTT) en **443** (HTTPS). De interne webdienst op poort 8000 en onversleuteld MQTT op poort 1883 blijven zo van buitenaf afgeschermd.

**Stap 1:** Firewallregels instellen en aanzetten  
Houd je huidige SSH-terminal open. We blokkeren eerst standaard alle inkomend verkeer (`deny incoming`), geven alléén de drie poorten hierboven vrij (`limit 22/tcp` remt ook herhaalde foute SSH-inlogpogingen af) en zetten daarna pas `ufw` aan. Bevestig de vraag `Proceed with operation (y|n)?` met **`y`** en druk op **Enter**:

</div>

```bash
sudo ufw default deny incoming
sudo ufw default allow outgoing
sudo ufw limit 22/tcp
sudo ufw allow 8883/tcp
sudo ufw allow 443/tcp
sudo ufw enable
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Stap 2:** Firewallregels en luisterende poorten controleren  
Controleer nu met onderstaande twee opdrachten zowel de actieve firewallregels (`ufw status numbered`) als welke programma's op welke poorten luisteren (`ss -lntp`):

</div>

```bash
sudo ufw status numbered
sudo ss -lntp
```

![Firewallregels en luisterende poorten op de Droplet](../afbeeldingen/schermafbeeldingen/03_firewall_poorten.png)

*Figuur 10. Voorbeelduitvoer van `ufw status numbered` en `ss -lntp` na het inschakelen van de firewall.*

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Wat zie je op deze uitvoer?**
- **Bovenaan (`ufw status numbered`):** `Status: active`, met alleen **`22/tcp`** (`LIMIT IN`), **`8883/tcp`** (`ALLOW IN`) en **`443/tcp`** (`ALLOW IN`) open voor inkomend verkeer (en dezelfde drie regels voor IPv6). Alle andere poorten worden door de firewall geblokkeerd.
- **Onderaan (`ss -lntp`):**
  - **`0.0.0.0:22`** (`sshd`), **`0.0.0.0:443`** (`nginx`) en **`0.0.0.0:8883`** (`mosquitto`) luisteren op `0.0.0.0`, wat betekent dat ze **van buitenaf via internet** bereikbaar zijn.
  - **`127.0.0.1:8000`** (`uvicorn`, onze Python-statusdienst) luistert uitsluitend op **`127.0.0.1`** (alleen binnen de server zelf), zodat externe aanvragen altijd eerst langs de beveiligde HTTPS-voordeur van Nginx (poort 443) moeten.

Open nu voor alle zekerheid een nieuwe terminal (in Codespaces of lokaal) en meld je nog een keer aan als **`beheer`** om te testen dat je nog steeds via SSH binnenkomt. Sluit je eerste verbinding pas nadat die nieuwe aanmelding werkt.

</div>

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Tip: een tweede firewall bij DigitalOcean</strong><br>Als het team ook een DigitalOcean Cloud Firewall aan je Droplet koppelt, moet die eveneens TCP 22, 8883 en 443 toelaten vanaf de afgesproken clientadressen. Codespaces-adressen kunnen veranderen. Laat de docent deze centrale regels controleren en verander geen firewall die ook servers van anderen beschermt.</div>

## 13. De verbinding vanuit Codespaces testen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Terminal, zonder SSH-verbinding)

Omdat we ons servercertificaat zelf hebben ondertekend (en niet bij een bekende certificaatinstantie hebben gekocht), kent Codespaces dat certificaat nog niet. Kopieer daarom het openbare `ca.crt` via SCP naar `gegevens/werk/` in Codespaces, zodat je Python-scripts straks zeker weten dat ze met jouw server spreken.

Voer de onderstaande cel uit en voer de afgedrukte opdracht uit in je Codespaces-terminal vanuit de hoofdmap van je repository. Vul jouw **`SSH beheer-wachtwoord`** in:

</div>

In [11]:
# Codespaces (Linux bash)
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}:/etc/iot/ca.crt gegevens/werk/ca.crt")

scp -o HostKeyAlgorithms=ssh-ed25519 beheer@178.62.241.168:/etc/iot/ca.crt gegevens/werk/ca.crt


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

*(Werk je toch lokaal op macOS of Windows in plaats van in Codespaces? Gebruik dan één van de twee cellen hieronder:)*

</div>

In [37]:
# Alleen lokaal op macOS (Terminal)
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}:/etc/iot/ca.crt gegevens/werk/ca.crt")

scp -o HostKeyAlgorithms=ssh-ed25519 beheer@178.62.246.100:/etc/iot/ca.crt gegevens/werk/ca.crt


In [13]:
# Alleen lokaal op Windows (PowerShell of Git Bash)
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}:/etc/iot/ca.crt gegevens\\werk\\ca.crt")

scp -o HostKeyAlgorithms=ssh-ed25519 beheer@64.225.71.10:/etc/iot/ca.crt gegevens\werk\ca.crt


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 03)

**Hoe komt de code aan je wachtwoorden?**  
De code bewaart nergens wachtwoorden op schijf. Wanneer je de cel hieronder uitvoert, vraagt `getpass(...)` jou twee keer om een wachtwoord **in een klein invoerbalkje helemaal bovenaan in het venster** van Codespaces/Cursor (niet onder de cel zelf, en tijdens het typen zie je geen tekens):
1. Eerst vraagt hij om jouw **`MQTT-broker eigenaar`**-wachtwoord (uit Stap 10) → typ of plak het en druk op **Enter**.
2. Daarna vraagt hij om jouw **`API-Droplet wachtwoord`** (uit Stap 11) → typ of plak het en druk op **Enter**. *(Vul dus geen SSH-wachtwoord in.)*

**Let op de wachttijd:** het uitvoeren van deze cel kan even duren (soms zelfs enkele minuten, zolang je bovenaan nog moet invullen of bij een tragere netwerkverbinding). Wacht rustig tot de uitvoer verschijnt en onderbreek de cel niet meteen.

**Wat gebeurt er onder water in `controleer(...)`?**  
Zodra je beide wachtwoorden hebt ingevoerd, test de aangeleverde functie uit `programmatuur/servercontrole.py` jouw server van buitenaf:
- Ze controleert met `gegevens/werk/ca.crt` het certificaat en IPv4-adres van jouw server.
- Ze meldt zich met account `eigenaar` en jouw **`MQTT-broker eigenaar`**-wachtwoord aan op poort **8883**, abonneert zich op een tijdelijk controletopic, stuurt één testbericht heen en controleert of precies datzelfde bericht weer terugkomt.
- Ze vraagt via HTTPS (poort **443**) met jouw **`API-Droplet wachtwoord`** de testroute `/gezondheid` op en controleert het antwoord.
- Direct na afloop wist het `finally:`-blok beide wachtwoorden weer uit het geheugen (`del`), zonder iets op te slaan of af te drukken.

</div>

In [12]:
from getpass import getpass
from datetime import datetime, timezone

sys.path.insert(0, str(hoofdmap / "programmatuur"))
from servercontrole import controleer

verbindingsresultaat = {}
mqtt_wachtwoord = api_wachtwoord = None
try:
    mqtt_wachtwoord = getpass("Wachtwoord 'MQTT-broker eigenaar': ")
    api_wachtwoord = getpass("Wachtwoord 'API-Droplet wachtwoord': ")
    verbindingsresultaat = controleer(serveradres, deelnemer, werkmap / "ca.crt", mqtt_wachtwoord, api_wachtwoord)
finally:
    del mqtt_wachtwoord, api_wachtwoord
print(json.dumps(verbindingsresultaat, indent=2))

{
  "mqtt_tls": "geslaagd",
  "mqtt_heen_en_terug": "geslaagd",
  "https": "geslaagd",
  "api_aanmelding": "geslaagd"
}


<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Controle</strong><br>Je verwacht vier keer <code>geslaagd</code>. Daarmee heb je vanaf deze werkplek aangetoond dat MQTT-aanmelding, publiceren en ontvangen, certificaatcontrole en de beveiligde webaanmelding werken. De verwerking en opslag van echte meetrecords volgen in de volgende notebooks.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 5.** Sluit alleen je SSH-terminal met exit en voer de verbindingstest opnieuw uit. Werkt die nog? Verklaar de uitkomst.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 5):**

Nee als je deze sluit kan niemand meer verbinding maken met je server

</div>

## 14. Zelf starten na een serverherstart

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal, als beheer)

Stel dat jouw Droplet bij DigitalOcean op een later moment opnieuw moet opstarten, bijvoorbeeld door onderhoud, een storing of na updates. De virtuele computer zelf start dan vanzelf weer op, maar omdat we Mosquitto bewust op `disabled` hebben gezet (en in Stap 4.6 geen automatische *Startup scripts* hebben toegevoegd), blijft de MQTT-broker uitstaan totdat je hem zelf opnieuw aanzet.

*(In een echte productie-omgeving zou je dit anders aanpakken: daar zet je de broker wel op automatisch starten (`enable`), of laat je via een opstartscript meteen bij het opstarten controleren op software-updates en veiligheidspatches voordat alle diensten starten. In dit practicum houden we het starten van Mosquitto handmatig, zodat je zelf leert controleren of hij actief is voordat andere gebruikers verbinden.)*

**Stap 1:** De server gecontroleerd herstarten  
Boot je Droplet één keer opnieuw op om precies dat scenario nu te testen:

</div>

```bash
sudo reboot
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Stap 2:** Opnieuw inloggen en controleren wat wel en niet is meegestart  
Door `sudo reboot` wordt je SSH-verbinding verbroken. Wacht ongeveer een halve minuut totdat de Droplet opnieuw is opgestart en meld je weer aan als `beheer` (met jouw **`SSH beheer-wachtwoord`**). Controleer daarna met onderstaande vier opdrachten de toestand van Mosquitto (`is-active` en `is-enabled`), de twee webdiensten (`iot-status` en `nginx`) en de firewall (`ufw`):

</div>

```bash
sudo systemctl is-active mosquitto
sudo systemctl is-enabled mosquitto
sudo systemctl is-active iot-status nginx
sudo ufw status
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Verwachte uitvoer:** voor Mosquitto zie je **`inactive`** en **`disabled`** (dat is hier goed, want hij staat bewust uit). Voor `iot-status` en `nginx` zie je twee keer **`active`** (die startten vanzelf mee), en de firewall toont **`Status: active`**.

**Stap 3:** Mosquitto zelf opnieuw starten  
Start de broker nu handmatig en bekijk zijn volledige status:

</div>

```bash
sudo systemctl start mosquitto
sudo systemctl status mosquitto --no-pager
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Verwachte uitvoer:** een groen bolletje (`●`) en **`Active: active (running)`**, terwijl bij `Loaded:` nog steeds **`disabled`** staat. Gebruik dit `start`-commando voortaan telkens na een serverherstart voordat je berichten verstuurt of andere gebruikers laat verbinden. *(Wil je de broker ooit zelf stoppen, gebruik dan `sudo systemctl stop mosquitto`, dan blijven al je configuratie, wachtwoorden en certificaten bewaard.)*

**Stap 4:** De verbindingstest nog een keer uitvoeren (Codespaces, Notebook 03)  
Ga terug naar **Stap 13** en voer de verbindingstest-cel nog een keer uit om te controleren dat zowel MQTT als HTTPS na de herstart weer helemaal werken. Noteer daarna de uitkomst in Vraag 6.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 6.** Welke toestand toont Mosquitto direct na de serverherstart en na jouw startopdracht? Wat geeft de verbindingstest daarna? Verklaar waarom `disabled` en `active` tegelijk kunnen voorkomen.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 6):**

Inactive en disabled juist na de herstart, na opdracht active. De verbindingstest slaagt omdat hij draait. Disabled betekent dat hij niet automatisch start na een reboot, maar als je hem zelf opstart is hij active.

</div>

## 15. Je resultaat bewaren en afsluiten

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 03)

Bewaar een klein controlebestand zodat je later ziet welke verbindingen je hebt getest. Het bestand bevat je deelnemerscode, testtijd en uitkomsten. Wachtwoorden, privésleutels en je serveradres horen niet in dit inleverbestand.

</div>

In [13]:
if not all(verbindingsresultaat.get(k) == "geslaagd" for k in ("mqtt_tls", "mqtt_heen_en_terug", "https", "api_aanmelding")):
    raise RuntimeError("Los eerst de mislukte verbindingstest op.")
verslag = {"deelnemerscode": deelnemer, "branch": branch,
           "getest_op_utc": datetime.now(timezone.utc).isoformat(),
           "verbindingen": verbindingsresultaat}
inleverpad = hoofdmap / "gegevens/inlevering/03_servercontrole.json"
inleverpad.parent.mkdir(parents=True, exist_ok=True)
inleverpad.write_text(json.dumps(verslag, indent=2) + "\n", encoding="utf-8")
print("Bewaard:", inleverpad.relative_to(hoofdmap))
print("Bewaar je eigen antwoorden over de service- en serverherstart in het notebook.")

Bewaard: gegevens/inlevering/03_servercontrole.json
Bewaar je eigen antwoorden over de service- en serverherstart in het notebook.


<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Stop je Codespace na het practicum</strong><br>Stop de Codespace na het bewaren van je werk om onnodig gebruik van de beschikbare uren te vermijden. De DigitalOcean-Droplet heeft een afzonderlijke facturering en blijft beschikbaar voor de vervolgopdrachten. Verwijder die pas volgens de afspraak met de docent.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Source Control) en op de GitHub-website

1. Vul je eigen antwoorden aan. Gebruik **File → Save All** vóór je wijzigingen selecteert.
2. Controleer de notebookuitvoer. Verwijder eventuele onverwacht zichtbare geheimen vóór een commit. De tijdelijke uitvoer met je serveradres kun je leegmaken, want het adres blijft in `gegevens/werk/server.json` bewaard.
3. Selecteer uitsluitend je aangepaste notebook en `gegevens/inlevering/03_servercontrole.json`. Gebruik een duidelijke commitboodschap en **Push** naar je eigen branch.
4. Controleer op GitHub je branch en de opgeslagen bestanden.
5. Bewaar je servernaam en wachtwoorden in je wachtwoordmanager. `server.json` en `ca.crt` blijven in de bestaande Codespace staan. Bij een nieuwe Codespace vul je het adres opnieuw in en haal je het openbare certificaat opnieuw op via gecontroleerde SSH.
6. Sluit de SSH-terminal met `exit`. Stop daarna de Codespace via [github.com/codespaces](https://github.com/codespaces). Alleen het browsertabblad sluiten is onvoldoende.

Voor de volgende sessie hervat je dezelfde Codespace en dezelfde Droplet. Is de server intussen herstart, meld je dan aan als beheer en voer `sudo systemctl start mosquitto` uit vóór de verbindingstest. Alleen je Codespace stoppen of je SSH-terminal sluiten stopt de broker niet. Het opnieuw aanmaken van de omgevingen is niet nodig.

</div>

## Hulp bij foutmeldingen

| Waarneming | Controle en herstel |
|---|---|
| Team of project ontbreekt | Controleer je account, teamuitnodiging en geselecteerde team, en laat de docent je rechten nakijken. |
| SSH geeft Permission denied | Gebruik na de eerste inrichting gebruiker `beheer` en jouw **`SSH beheer-wachtwoord`**. Directe root-aanmelding is uitgeschakeld. |
| `client_loop: send disconnect: Connection reset` (SSH verbroken) | De SSH-sessie viel weg (bijvoorbeeld na even stilstaan of slaapstand). Druk in je terminal op **Pijl omhoog (`↑`)** en **Enter** (of plak de `ssh`-opdracht als `beheer` opnieuw) en meld je opnieuw aan. Wat je al op de server hebt ingesteld blijft gewoon bewaard. Voeg zo nodig `-o ServerAliveInterval=30` toe om de sessie wakker te houden. |
| SSH geeft een gewijzigde host key | Vergelijk opnieuw met de vingerafdruk in de DigitalOcean-console. Accepteer geen vervanging op basis van alleen de foutmelding. |
| Verbinding krijgt een time-out | Controleer of je eigen Droplet draait en of het IPv4-adres, UFW en eventuele Cloud Firewall kloppen. |
| Certificaatcontrole mislukt | Controleer serveradres, datum/tijd, vervaldatum en de via SSH opgehaalde `ca.crt`. Schakel certificaatcontrole niet uit. |
| MQTT geeft Connection refused na een serverherstart | Start de broker op de server met `sudo systemctl start mosquitto` en controleer zijn status. |
| MQTT-aanmelding geweigerd | Gebruik account `eigenaar` en jouw **`MQTT-broker eigenaar`**-wachtwoord, niet je SSH- of API-wachtwoord. |
| Controlebericht komt niet terug | Controleer de deelnemerscode in `practicum.rechten` en notebook 01. Herstart Mosquitto na het herstellen van zijn configuratie. |
| HTTPS geeft 401 | De verbinding werkt, maar de API-aanmelding is ongeldig. Gebruik jouw **`API-Droplet wachtwoord`**. |
| HTTPS geeft 502 | Nginx draait, maar de interne statusdienst is niet bereikbaar. Controleer `iot-status`. |
| HTTPS geeft 429 | Er kwamen te veel aanvragen kort na elkaar. Wacht even en voer één controle uit. |
| Certificaatmap bestaat al | Bij dezelfde instellingen blijven bestaande certificaten behouden. Bij een ander adres of onvolledige map vraag je hulp. Overschrijf de sleutels niet. |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal of herstelconsole)

Lees bij een servicefout de laatste logregels. Deze opdrachten veranderen niets:

</div>

```bash
sudo journalctl -u mosquitto -n 30 --no-pager
sudo journalctl -u iot-status -n 30 --no-pager
sudo journalctl -u nginx -n 30 --no-pager
sudo tail -n 30 /var/log/mosquitto/mosquitto.log
sudo nginx -t
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Een padfout of `Permission denied` in een service wijst vaak op een ontbrekend bestand of onjuiste leesrechten. Controleer de bijbehorende stap. Plak geen wachtwoorden, privésleutels of volledige geheime configuratiebestanden in een antwoordcel of GitHub-bericht.

Bij verlies van je API-wachtwoord laat je de docent een nieuwe controlewaarde instellen en de statusdienst herstarten. Bij een verlopen servercertificaat of een nieuwe CA moeten het certificaat vernieuwd worden en moeten clients het nieuwe `ca.crt` ophalen.

</div>

## Technische bronnen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- [DigitalOcean: een Droplet aanmaken](https://docs.digitalocean.com/products/droplets/how-to/create/), [SSH](https://docs.digitalocean.com/products/droplets/how-to/connect-with-ssh/) en [Recovery Console](https://docs.digitalocean.com/products/droplets/how-to/recovery/recovery-console/).
- [Debian 13: Mosquitto-pakket](https://packages.debian.org/trixie/mosquitto) en [Python-pakket](https://packages.debian.org/trixie/python3).
- [Mosquitto: authenticatie](https://mosquitto.org/documentation/authentication-methods/) en [configuratie](https://mosquitto.org/man/mosquitto-conf-5.html). De lesconfiguratie gebruikt de Mosquitto 2.0-versie uit Debian 13. Instructies voor nieuwere hoofdversies kunnen verschillen.
- [Paho: MQTT-client](https://eclipse.dev/paho/files/paho.mqtt.python/html/client.html), [Nginx: HTTPS](https://nginx.org/en/docs/http/configuring_https_servers.html) en [FastAPI: HTTP Basic](https://fastapi.tiangolo.com/advanced/security/http-basic-auth/).
- [Bronnen van de schermafbeeldingen](../afbeeldingen/schermafbeeldingen/BRONNEN.md).

</div>